In [7]:
import requests, time, json
import pandas as pd

headers = {"x-api-version": "2", "Accept": "application/json"}

# --- Get the list of London boroughs (same phonebook idea as before, JSON version) ---
r = requests.get("https://api.ratings.food.gov.uk/Authorities", headers=headers)


authorities = r.json()["authorities"]                # the list of all councils

test = pd.json_normalize(authorities)

test[test['RegionName'] == 'London']

,LocalAuthorityId,LocalAuthorityIdCode,Name,FriendlyName,Url,SchemeUrl,Email,RegionName,FileName,FileNameWelsh,EstablishmentCount,CreationDate,LastPublishedDate,SchemeType,links
14,88,501,Barking and Dagenham,barking-and-dagenham,http://www.lbbd.gov.uk/Pages/Home.aspx,,foodsafety@lbbd.gov.uk,London,https://ratings.food.gov.uk/OpenDataFiles/FHRS...,None,1448,2010-08-17T15:30:24.87,2026-07-16T00:32:05.773,1,"[{'rel': 'self', 'href': 'https://api.ratings...."
15,89,502,Barnet,barnet,http://www.barnet.gov.uk/,,FoodSafety@barnet.gov.uk,London,https://ratings.food.gov.uk/OpenDataFiles/FHRS...,None,2867,2010-08-17T15:30:24.87,2026-07-18T00:33:33.597,1,"[{'rel': 'self', 'href': 'https://api.ratings...."
23,90,503,Bexley,bexley,http://www.bexley.gov.uk,,food.safety@bexley.gov.uk,London,https://ratings.food.gov.uk/OpenDataFiles/FHRS...,None,1766,2010-08-17T15:30:24.87,2026-07-18T00:33:41.413,1,"[{'rel': 'self', 'href': 'https://api.ratings...."
37,91,504,Brent,brent,http://www.brent.gov.uk/,,ens.foodsafety@brent.gov.uk,London,https://ratings.food.gov.uk/OpenDataFiles/FHRS...,None,2488,2010-08-17T15:30:24.87,2026-07-18T00:33:53.607,1,"[{'rel': 'self', 'href': 'https://api.ratings...."
43,92,505,Bromley,bromley,http://www.bromley.gov.uk,,food@bromley.gov.uk,London,https://ratings.food.gov.uk/OpenDataFiles/FHRS...,None,2459,2010-08-17T15:30:24.87,2026-07-18T00:34:08.64,1,"[{'rel': 'self', 'href': 'https://api.ratings...."
53,93,506,Camden,camden,http://www.camden.gov.uk,,foodsafety@camden.gov.uk,London,https://ratings.food.gov.uk/OpenDataFiles/FHRS...,None,4275,2010-08-17T15:30:24.87,2026-07-18T00:34:30.79,1,"[{'rel': 'self', 'href': 'https://api.ratings...."
71,95,508,City of London Corporation,city-of-london-corporation,http://www.cityoflondon.gov.uk/Corporation/hom...,,publicprotection@cityoflondon.gov.uk,London,https://ratings.food.gov.uk/OpenDataFiles/FHRS...,None,1801,2010-08-17T15:30:24.87,2026-07-18T00:35:02.847,1,"[{'rel': 'self', 'href': 'https://api.ratings...."
80,94,507,Croydon,croydon,http://www.croydon.gov.uk,,food.safety@croydon.gov.uk,London,https://ratings.food.gov.uk/OpenDataFiles/FHRS...,None,3146,2010-08-17T15:30:24.87,2026-07-18T00:34:51.993,1,"[{'rel': 'self', 'href': 'https://api.ratings...."
96,96,509,Ealing,ealing,https://www.ealing.gov.uk/info/201150/food_hyg...,,foodsafety@ealing.gov.uk,London,https://ratings.food.gov.uk/OpenDataFiles/FHRS...,None,3670,2010-08-17T15:30:24.87,2026-07-18T00:35:18.607,1,"[{'rel': 'self', 'href': 'https://api.ratings...."
113,97,510,Enfield,enfield,http://www.enfield.gov.uk/,,environmental.health@enfield.gov.uk,London,https://ratings.food.gov.uk/OpenDataFiles/FHRS...,None,2376,2010-08-17T15:30:24.87,2026-07-18T00:35:32.667,1,"[{'rel': 'self', 'href': 'https://api.ratings...."


In [11]:
london = []
for a in authorities:
    if a["RegionName"] == "London":                  # keep London ones only
        london.append(a)
print(f'borough = {len(london)}')                                  

borough = 33


In [12]:
# --- Download establishments for every borough ---
all_records = []

for la in london:
    # first page: also tells us how many pages this borough has
    params = {"localAuthorityId": la["LocalAuthorityId"], "pageSize": 1000, "pageNumber": 1}
    data = requests.get("https://api.ratings.food.gov.uk/Establishments",
                        headers=headers, params=params).json()

    total_pages = data["meta"]["totalPages"]         # the API tells us directly this time

    for page in range(1, total_pages + 1):           # go through every page
        params["pageNumber"] = page                  # same request, different page number
        data = requests.get("https://api.ratings.food.gov.uk/Establishments",
                            headers=headers, params=params).json()
        for rec in data["establishments"]:
            rec["borough"] = la["Name"]              # stamp which borough it came from
            all_records.append(rec)
        time.sleep(0.5)                              # polite pause

    print(la["Name"], "done — running total:", len(all_records))

Barking and Dagenham done — running total: 1448
Barnet done — running total: 4315
Bexley done — running total: 6081
Brent done — running total: 8569
Bromley done — running total: 11028
Camden done — running total: 15303
City of London Corporation done — running total: 17104
Croydon done — running total: 20250
Ealing done — running total: 23920
Enfield done — running total: 26296
Greenwich done — running total: 28678
Hackney done — running total: 31294
Hammersmith and Fulham done — running total: 33272
Haringey done — running total: 35230
Harrow done — running total: 37114
Havering done — running total: 37871
Hillingdon done — running total: 40107
Hounslow done — running total: 42405
Islington done — running total: 44992
Kensington and Chelsea done — running total: 47004
Kingston-Upon-Thames done — running total: 48422
Lambeth done — running total: 51052
Lewisham done — running total: 53526
Merton done — running total: 55073
Newham done — running total: 57735
Redbridge done — running to

In [ ]:
'''# --- Save, then load into pandas ---
with open("fsa_london.json", "w") as f:
    json.dump(all_records, f)
 '''
